<a href="https://colab.research.google.com/github/futuregunmulju/2026-Signals-and-Systems-Speech-Noise-Reduction/blob/main/%5B9%EC%A1%B0%5D_Challenge1_Speech_Noise_Reduction_Algorithms.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -q "numpy>=1.26,<3" "scipy>=1.12,<2" "soundfile>=0.12,<1" "pandas>=2.1,<4" "tqdm>=4.66,<5" "pesq==0.0.4"

from google.colab import drive
drive.mount("/content/drive")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
Mounted at /content/drive


In [3]:
from pathlib import Path
import unicodedata

import numpy as np
import pandas as pd
import soundfile as sf
from scipy import signal
from scipy.ndimage import uniform_filter1d
from scipy.special import i0, i1, exp1
from pesq import pesq
from tqdm.auto import tqdm

# 기존 Challenge 1 노트북과 동일한 폴더
folder = "/content/drive/MyDrive/연세대/신호및시스템/9조_noise_reduction_challenge"
DATA_ROOT = Path(folder)

# 기존 노트북에 저장된 한글 경로의 유니코드 표기도 지원
if not DATA_ROOT.is_dir():
    DATA_ROOT = Path(unicodedata.normalize("NFD", folder))

SR = 16000
validation = pd.read_csv(DATA_ROOT / "validation" / "samples.csv")


def read_validation_pair(row):
    clean, clean_sr = sf.read(
        DATA_ROOT / row.clean_path, dtype="float64"
    )
    noisy, noisy_sr = sf.read(
        DATA_ROOT / row.noisy_path, dtype="float64"
    )

    if clean_sr != SR or noisy_sr != SR:
        raise ValueError(f"{row.segment_id}: 16 kHz 음성이 필요합니다.")
    if clean.shape != (48000,) or noisy.shape != clean.shape:
        raise ValueError(f"{row.segment_id}: 3초 모노 음성이 필요합니다.")
    if not np.isfinite(clean).all() or not np.isfinite(noisy).all():
        raise ValueError(f"{row.segment_id}: 유효하지 않은 음성 값입니다.")

    return clean, noisy


print(f"Validation: {len(validation)}쌍")

Validation: 164쌍


In [4]:
N_FFT = 512
HOP = 128
NOISE_FRACTION = 0.20
SMOOTHING = 0.98
GAIN_FLOOR = 0.05


def stft_audio(audio):
    return signal.stft(
        audio,
        fs=SR,
        window="hann",
        nperseg=N_FFT,
        noverlap=N_FFT - HOP,
        nfft=N_FFT,
        boundary="zeros",
        padded=True,
    )


def inverse_stft(spectrum, length):
    _, audio = signal.istft(
        spectrum,
        fs=SR,
        window="hann",
        nperseg=N_FFT,
        noverlap=N_FFT - HOP,
        nfft=N_FFT,
        input_onesided=True,
        boundary=True,
    )
    if len(audio) < length or not np.isfinite(audio).all():
        raise ValueError("음성 복원 결과가 유효하지 않습니다.")
    return audio[:length]


def estimate_noise_power(spectrum, times, length):
    power = np.abs(spectrum) ** 2
    centers = np.rint(times * SR).astype(int)

    complete = np.flatnonzero(
        (centers >= N_FFT // 2)
        & (centers + N_FFT // 2 <= length)
    )
    if len(complete) == 0:
        complete = np.arange(power.shape[1])

    energy = power[:, complete].mean(axis=0)
    count = max(1, int(np.ceil(NOISE_FRACTION * len(complete))))
    selected = complete[np.argsort(energy, kind="stable")[:count]]
    noise = power[:, selected].mean(axis=1)

    return np.maximum(
        uniform_filter1d(noise, size=3, mode="nearest"),
        1e-18,
    )


def estimate_snr(spectrum, noise):
    power = np.abs(spectrum) ** 2
    gamma = np.maximum(power / noise, 1e-12)
    xi = np.maximum(gamma - 1.0, 0.0)

    for frame in range(1, spectrum.shape[1]):
        xi[:, frame] = (
            SMOOTHING
            * (np.abs(spectrum[:, frame - 1]) ** 2 / noise[:, 0])
            + (1.0 - SMOOTHING)
            * np.maximum(gamma[:, frame] - 1.0, 0.0)
        )

    v = np.maximum(gamma * xi / (1.0 + xi), 1e-12)
    return gamma, xi, v


def band_pass_filter(noisy):
    sos = signal.butter(
        4,
        (120.0, 3800.0),
        btype="bandpass",
        fs=SR,
        output="sos",
    )
    padlen = min(3 * (2 * len(sos) + 1), len(noisy) - 1)
    return signal.sosfiltfilt(sos, noisy, padlen=padlen)


def mmse_stsa_filter(noisy):
    _, times, spectrum = stft_audio(noisy)
    noise = estimate_noise_power(
        spectrum, times, len(noisy)
    )[:, None]

    gamma, xi, v = estimate_snr(spectrum, noise)
    half_v = np.minimum(v / 2.0, 700.0)

    gain = (
        np.sqrt(np.pi * v) / (2.0 * gamma)
        * np.exp(-half_v)
        * ((1.0 + v) * i0(half_v) + v * i1(half_v))
    )
    gain = np.clip(
        np.nan_to_num(gain, nan=0.0, posinf=1.0),
        GAIN_FLOOR,
        1.0,
    )
    return inverse_stft(gain * spectrum, len(noisy))


def mmse_lsa_filter(noisy):
    _, times, spectrum = stft_audio(noisy)
    noise = estimate_noise_power(
        spectrum, times, len(noisy)
    )[:, None]

    gamma, xi, v = estimate_snr(spectrum, noise)
    gain = (xi / (1.0 + xi)) * np.exp(0.5 * exp1(v))
    gain = np.clip(
        np.nan_to_num(gain, nan=0.0, posinf=1.0),
        GAIN_FLOOR,
        1.0,
    )
    return inverse_stft(gain * spectrum, len(noisy))


def denoise(noisy):
    enhanced = band_pass_filter(noisy)
    enhanced = mmse_stsa_filter(enhanced)
    enhanced = mmse_lsa_filter(enhanced)
    return enhanced

In [5]:
scores = []

for row in tqdm(
    validation.itertuples(index=False),
    total=len(validation),
    desc="Validation",
):
    clean, noisy = read_validation_pair(row)
    enhanced = denoise(noisy)

    valid = int(getattr(row, "valid_samples", len(clean)))
    if not 0 < valid <= len(clean):
        raise ValueError(
            f"{row.segment_id}: 잘못된 valid_samples={valid}"
        )

    scores.append(
        pesq(SR, clean[:valid], enhanced[:valid], "wb")
    )

print(
    f"\nBand-pass → MMSE-STSA → MMSE-LSA\n"
    f"Validation {len(scores)}쌍 평균 WB-PESQ: {np.mean(scores):.6f}"
)

Validation:   0%|          | 0/164 [00:00<?, ?it/s]

/tmp/ipykernel_3171/3626680540.py:100: RuntimeWarning: overflow encountered in multiply
  * ((1.0 + v) * i0(half_v) + v * i1(half_v))
/tmp/ipykernel_3171/3626680540.py:100: RuntimeWarning: overflow encountered in add
  * ((1.0 + v) * i0(half_v) + v * i1(half_v))



Band-pass → MMSE-STSA → MMSE-LSA
Validation 164쌍 평균 WB-PESQ: 1.612048
